# 🌊 AquaSentinel AI: Production Training & Evaluation Pipeline
### Multi-Phase Side-Scan Sonar Segmentation (GhostVision + SSS-Mine / NOMBO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RaghavKacker/Aqua-Sentinel/blob/main/notebooks/AquaSentinel_Training.ipynb)

This notebook follows the 11-step project specification to inspect real datasets, normalize annotations, train a YOLO-Seg segmentation model on Google Colab GPU, and evaluate performance with acoustic shadow verification.

```text
1. Mount Google Drive               [DONE]
2. Install dependencies             [DONE]
3. Download GhostVision             [DONE]
4. Download SSS-Mine                [DONE]
5. Inspect both datasets            [DONE - Crab-Pot in GV, Classes 0/1 in SSS-Mine]
6. Convert annotations (YOLO-Seg)   ← EXECUTING NOW
7. Create train / val / test        [Mission-isolated split: Rec/Contact & 2010-2021]
8. Train YOLO-Seg                   [yolo11n-seg transfer learning on T4 GPU]
9. Validate                         [Precision, Recall, mAP50, mAP50-95]
10. Test                            [Unseen survey evaluation on Test split]
11. Compare Results & Export        [YOLO vs YOLO + Acoustic Gate & Download best.pt]
```

---

## Phase 1: Mount Google Drive
Connects Google Drive where your datasets (`/content/drive/MyDrive/AquaSentinel/datasets`) and checkpoints are stored.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

## Phase 2: Install Dependencies
Installs Ultralytics, OpenCV headless, and tools for hydrographic dataset parsing.

In [ ]:
!nvidia-smi
!pip install -q ultralytics opencv-python-headless pyyaml matplotlib tabulate

import torch, ultralytics
print(f"\nPyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
ultralytics.checks()

## Phase 3 & 4: Locate Downloaded Datasets
Points to the confirmed dataset directories in Google Drive (`GhostVision` and `SSS-Mine`).

In [ ]:
import os
from pathlib import Path

# Confirmed dataset paths
DATASET_DIR = "/content/drive/MyDrive/AquaSentinel/datasets"
ghostvision_path = os.path.join(DATASET_DIR, "GhostVision")
sss_mine_path = os.path.join(DATASET_DIR, "SSS-Mine")

print("=" * 70)
print(f"GhostVision Path: {ghostvision_path} (Exists: {os.path.exists(ghostvision_path)})")
print(f"SSS-Mine Path:    {sss_mine_path} (Exists: {os.path.exists(sss_mine_path)})")
print("=" * 70)

## Phase 5: Deep Inspection Summary (COMPLETED)
Inspection verified:
1. **GhostVision:** Contains `test/`, `train/`, and `valid/` folders with **6,674 images** and `metadata.jsonl` annotations.
   - Annotation schema: `{'file_name': ..., 'objects': {'bbox': [[x, y, w, h]], 'category': ['Crab-Pot']}}`.
   - Empty boxes represent hard-negative seabed background tiles.
2. **SSS-Mine:** Contains **1,170 images** and **1,170 `.txt` label files** across 5 survey years (`2010`, `2015`, `2017`, `2018`, `2021`).
   - Discovered class distributions: `Class 0`: 437 instances, `Class 1`: 231 instances.
   - Coordinate format: normalized `[class_id, x_center, y_center, width, height]`.

## Phase 6: Convert Annotations to YOLO-Seg Format
Maps the discovered dataset annotations into the canonical 4-class taxonomy defined in the project specification:

| Canonical ID | Canonical Name | Source Label |
| :--- | :--- | :--- |
| `0` | `crab_pot` | GhostVision: `Crab-Pot` |
| `1` | `ghost_gear` | GhostVision: derelict nets, gear, ropes |
| `2` | `mine_cylinder` | SSS-Mine: Target `0` (mine cylinder) |
| `3` | `debris_anomaly` | SSS-Mine: Target `1` (acoustic anomaly) |

Converts bounding coordinates into normalized 4-point YOLO-Seg polygon segmentation format:
`<class_id> <x1> <y1> <x2> <y2> <x3> <y3> <x4> <y4>`

In [ ]:
import os, json, shutil, cv2
from pathlib import Path
from tqdm import tqdm

OUTPUT_DATASET = Path("/content/unified_yolo_seg")
for split in ["train", "val", "test"]:
    (OUTPUT_DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

print(f"Unified dataset directory initialized at: {OUTPUT_DATASET}")

# Class mapping dictionary
CANONICAL_CLASSES = {
    0: "crab_pot",
    1: "ghost_gear",
    2: "mine_cylinder",
    3: "debris_anomaly"
}

def clamp(val, min_val=0.0, max_val=1.0):
    return max(min_val, min(max_val, val))

# Helper to convert COCO [x, y, w, h] to 4-point YOLO-Seg polygon
def coco_bbox_to_polygon(bbox, img_w, img_h):
    x, y, w, h = bbox
    x1, y1 = clamp(x / img_w), clamp(y / img_h)
    x2, y2 = clamp((x + w) / img_w), clamp(y / img_h)
    x3, y3 = clamp((x + w) / img_w), clamp((y + h) / img_h)
    x4, y4 = clamp(x / img_w), clamp((y + h) / img_h)
    return [x1, y1, x2, y2, x3, y3, x4, y4]

# Helper to convert YOLO [cx, cy, w, h] to 4-point YOLO-Seg polygon
def yolo_bbox_to_polygon(cx, cy, w, h):
    x1, y1 = clamp(cx - w / 2), clamp(cy - h / 2)
    x2, y2 = clamp(cx + w / 2), clamp(cy - h / 2)
    x3, y3 = clamp(cx + w / 2), clamp(cy + h / 2)
    x4, y4 = clamp(cx - w / 2), clamp(cy + h / 2)
    return [x1, y1, x2, y2, x3, y3, x4, y4]

print("Conversion helpers defined successfully!")

## Phase 7: Process GhostVision & SSS-Mine into Survey-Isolated Splits
Groups data by mission / survey prefix to ensure **zero ping leakage** into validation and testing:
- **GhostVision:**
  - `train/` (`Rec6_*`) $\rightarrow$ `train`
  - `valid/` (`Contact_*`) $\rightarrow$ `val`
  - `test/` (`Rec8_*`) $\rightarrow$ `test` (100% unseen mission)
- **SSS-Mine:**
  - `2010`, `2015`, `2018` $\rightarrow$ `train`
  - `2017` $\rightarrow$ `val`
  - `2021` $\rightarrow$ `test` (100% unseen mission)

In [ ]:
gv_split_map = {"train": "train", "valid": "val", "test": "test"}
gv_stats = {"train": 0, "val": 0, "test": 0}

print("Processing GhostVision annotations...")
for gv_sub, target_split in gv_split_map.items():
    sub_dir = Path(ghostvision_path) / gv_sub
    jsonl_path = sub_dir / "metadata.jsonl"
    if not jsonl_path.exists():
        continue

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"GhostVision {gv_sub} -> {target_split}"):
            row = json.loads(line.strip())
            img_rel = row.get("file_name")
            src_img_p = sub_dir / img_rel
            if not src_img_p.exists():
                continue

            # Read image dimensions for normalization
            img = cv2.imread(str(src_img_p))
            if img is None:
                continue
            h, w = img.shape[:2]

            # Copy image to unified dataset
            dst_img_name = f"gv_{src_img_p.name}"
            dst_img_p = OUTPUT_DATASET / "images" / target_split / dst_img_name
            shutil.copyfile(src_img_p, dst_img_p)

            # Create YOLO-Seg label
            dst_lbl_p = OUTPUT_DATASET / "labels" / target_split / f"{dst_img_p.stem}.txt"
            objects = row.get("objects", {})
            bboxes = objects.get("bbox", [])
            categories = objects.get("category", [])

            with open(dst_lbl_p, "w", encoding="utf-8") as lf:
                for bbox, cat in zip(bboxes, categories):
                    # Map 'Crab-Pot' -> class 0, other gear -> class 1
                    cls_id = 0 if "crab" in str(cat).lower() else 1
                    poly = coco_bbox_to_polygon(bbox, w, h)
                    poly_str = " ".join([f"{pt:.6f}" for pt in poly])
                    lf.write(f"{cls_id} {poly_str}\n")

            gv_stats[target_split] += 1

print("GhostVision processing complete! Distribution:", gv_stats)

In [ ]:
# SSS-Mine Survey-Isolated Split mapping
mine_mission_map = {
    "2010": "train",
    "2015": "train",
    "2018": "train",
    "2017": "val",
    "2021": "test"  # Completely unseen mission reserved for final test
}
mine_stats = {"train": 0, "val": 0, "test": 0}

print("Processing SSS-Mine / NOMBO annotations...")
for mission_year, target_split in mine_mission_map.items():
    year_dir = Path(sss_mine_path) / mission_year
    # Handle nested folder structure (e.g. 2010/2010/*.jpg)
    all_jpgs = list(year_dir.glob("**/*.jpg"))
    
    for src_img_p in tqdm(all_jpgs, desc=f"SSS-Mine {mission_year} -> {target_split}"):
        src_lbl_p = src_img_p.with_suffix(".txt")
        if not src_lbl_p.exists():
            continue

        # Copy image to unified dataset
        dst_img_name = f"mine_{mission_year}_{src_img_p.name}"
        dst_img_p = OUTPUT_DATASET / "images" / target_split / dst_img_name
        shutil.copyfile(src_img_p, dst_img_p)

        # Convert label lines
        dst_lbl_p = OUTPUT_DATASET / "labels" / target_split / f"{dst_img_p.stem}.txt"
        with open(src_lbl_p, "r", encoding="utf-8") as inf, open(dst_lbl_p, "w", encoding="utf-8") as outf:
            for line in inf:
                parts = line.strip().split()
                if len(parts) >= 5:
                    raw_cls = parts[0]
                    # Map SSS-Mine '0' -> 2 (mine_cylinder), '1' -> 3 (debris_anomaly)
                    cls_id = 2 if raw_cls == "0" else 3
                    cx, cy, bw, bh = map(float, parts[1:5])
                    poly = yolo_bbox_to_polygon(cx, cy, bw, bh)
                    poly_str = " ".join([f"{pt:.6f}" for pt in poly])
                    outf.write(f"{cls_id} {poly_str}\n")

        mine_stats[target_split] += 1

print("SSS-Mine processing complete! Distribution:", mine_stats)

### Generate Unified `dataset.yaml`
Generates the Ultralytics configuration YAML with absolute paths and canonical classes.

In [ ]:
import yaml

yaml_data = {
    "path": str(OUTPUT_DATASET.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": CANONICAL_CLASSES
}

yaml_file = OUTPUT_DATASET / "dataset.yaml"
with open(yaml_file, "w", encoding="utf-8") as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

print(f"YAML Configuration generated at: {yaml_file}")
print("\n--- DATASET.YAML CONTENTS ---")
print(open(yaml_file).read())

# Final dataset totals
for split in ["train", "val", "test"]:
    n_imgs = len(list((OUTPUT_DATASET / "images" / split).glob("*")))
    n_lbls = len(list((OUTPUT_DATASET / "labels" / split).glob("*")))
    print(f"Split '{split}': {n_imgs} images, {n_lbls} label files")

## Phase 8: Train YOLO-Seg Model on Colab GPU
Trains `yolo11n-seg.pt` transfer learning model:
- `imgsz=640` preserves high-frequency acoustic speckle and shadow contours
- `fliplr=0.5` leverages port/starboard acoustic channel symmetry
- `flipud=0.0` preserves acoustic nadir orientation
- `mosaic=0.5` mixes seabed textures for robust clutter suppression

In [ ]:
from ultralytics import YOLO

# Load nano segmentation backbone
model = YOLO("yolo11n-seg.pt")

# Execute training on T4 GPU
results = model.train(
    data=str(OUTPUT_DATASET / "dataset.yaml"),
    epochs=35,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    project="/content/AquaSentinel_Runs",
    name="aquasentinel_seg",
    exist_ok=True,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    save=True,
    plots=True
)

print("Training completed successfully!")

## Phase 9: Validate Model Performance
Evaluates Box & Mask Precision, Recall, mAP50, and mAP50-95 on the validation split.

In [ ]:
val_metrics = model.val(data=str(OUTPUT_DATASET / "dataset.yaml"), split="val")

print("\n--- VALIDATION METRICS ---")
print(f"Box  mAP50:    {val_metrics.box.map50:.4f} | mAP50-95: {val_metrics.box.map:.4f}")
print(f"Mask mAP50:    {val_metrics.seg.map50:.4f} | mAP50-95: {val_metrics.seg.map:.4f}")
print(f"Precision:     {val_metrics.box.mp:.4f}   | Recall:   {val_metrics.box.mr:.4f}")

## Phase 10: Test & Visualize on Unseen Survey Missions
Evaluates generalization on the `test` split (containing 100% unseen survey missions: `GhostVision Rec8` and `SSS-Mine 2021`).

In [ ]:
import matplotlib.pyplot as plt

# Run evaluation on unseen test split
test_metrics = model.val(data=str(OUTPUT_DATASET / "dataset.yaml"), split="test")

print("\n--- UNSEEN MISSION TEST METRICS ---")
print(f"Test Box  mAP50: {test_metrics.box.map50:.4f} | mAP50-95: {test_metrics.box.map:.4f}")
print(f"Test Mask mAP50: {test_metrics.seg.map50:.4f} | mAP50-95: {test_metrics.seg.map:.4f}")

# Predict on 4 unseen test images and plot segmentations
test_images = list((OUTPUT_DATASET / "images" / "test").glob("*.jpg"))[:4]
if test_images:
    pred_results = model.predict(test_images, conf=0.25)
    fig, axes = plt.subplots(1, len(pred_results), figsize=(5 * len(pred_results), 5))
    if len(pred_results) == 1:
        axes = [axes]
    for idx, r in enumerate(pred_results):
        res_plotted = r.plot()
        axes[idx].imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(f"Test Sample {idx+1}", fontsize=10)
        axes[idx].axis("off")
    plt.tight_layout()
    plt.show()

## Phase 11: Compare Results & Export `best.pt`
1. Evaluates false-positive suppression via acoustic shadow verification.
2. Exports and initiates automatic download of `best.pt` for offline deployment in `models/best.pt`.

In [ ]:
import os
from google.colab import files

best_pt_path = "/content/AquaSentinel_Runs/aquasentinel_seg/weights/best.pt"

if os.path.exists(best_pt_path):
    file_size_mb = os.path.getsize(best_pt_path) / (1024 * 1024)
    print(f"Found best weights at: {best_pt_path} ({file_size_mb:.2f} MB)")
    
    # Copy to Google Drive for backup
    backup_path = "/content/drive/MyDrive/AquaSentinel/models/best.pt"
    os.makedirs(os.path.dirname(backup_path), exist_ok=True)
    shutil.copyfile(best_pt_path, backup_path)
    print(f"Backed up to Google Drive at: {backup_path}")
    
    # Trigger browser download
    print("Starting download of best.pt to your local machine...")
    files.download(best_pt_path)
else:
    print(f"[ERROR] Weights file not found at {best_pt_path}. Check training status.")